# ROI бедра

Проверяет эвристику по высоте кадра. Нейросеть для ROI не обучается. Метки берутся из `femur_img_labels.csv`. Скор — минус высота кадра: чем кадр короче, тем выше скор. Операционная точка: `Rows <= 210`, то есть `height_mm <= 220.5`.

AUC здесь только ранжирование. Позитивов 7, поэтому это не клиническая оценка.


## Данные


In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

import dxa_utils as U

df = pd.read_csv("femur_img_labels.csv")
df = df[(df["femur_side"].isin(["L", "R"])) & df["femur_roi"].notna()].copy()
df["femur_roi"] = df["femur_roi"].astype(int)
df["height_mm"] = df["height"] * U.PX_Y_MM
print("images", len(df), "studies", df.study_id.nunique(), "positives", int(df.femur_roi.sum()))
print(df.groupby("femur_roi")["height"].describe())


images 148 studies 77 positives 7
           count        mean        std    min    25%    50%    75%    max
femur_roi                                                                 
0          141.0  276.680851  34.207626  207.0  261.0  265.0  292.0  349.0
1            7.0  214.857143  25.829109  180.0  206.0  207.0  222.0  261.0


## Ранжирование


In [2]:
score = -df["height_mm"].values
y = df["femur_roi"].values
auc = roc_auc_score(y, score)
print(f"Image-level AUC (descriptive): {auc:.3f}")


def auc_stat(idx):
    yy = y[idx]
    ss = score[idx]
    if len(np.unique(yy)) < 2:
        return np.nan
    return roc_auc_score(yy, ss)


point, lo, hi, n_boot = U.cluster_bootstrap(
    df["study_id"].values, auc_stat, n_boot=2000, seed=42
)
print(f"Cluster-bootstrap AUC: {point:.3f} [{lo:.3f}, {hi:.3f}] (valid boots={n_boot})")


Image-level AUC (descriptive): 0.938
Cluster-bootstrap AUC: 0.938 [0.824, 1.000] (valid boots=1995)


## Операционная точка


In [3]:
THRESH_PX = 210
THRESH_MM = THRESH_PX * U.PX_Y_MM
pred = (df["height"].values <= THRESH_PX).astype(int)
print("Fixed threshold:", THRESH_PX, "px", THRESH_MM, "mm")
print(U.binary_metrics(y, pred))
print("Flagged share:", pred.mean())


Fixed threshold: 210 px 220.5 mm
{'sens': 0.7142857142857143, 'spec': 0.9858156028368794, 'precision': 0.7142857142857143, 'f1': 0.7142857142857143, 'bal_acc': np.float64(0.8500506585612968), 'tp': 5, 'fp': 2, 'fn': 2, 'tn': 139}
Flagged share: 0.0472972972972973


## Порог по исследованиям

Для каждого исследования порог подбирается по F1 на остальных. При семи позитивах этот порог почти не двигается, но ошибка на отложенном исследовании ниже, чем на фиксированных 210 px. Это описание устойчивости, не отдельный тест.


In [4]:
thresholds = []
loo_rows = []
for study, val in df.groupby("study_id"):
    train = df[df.study_id != study]
    yy = train.femur_roi.values
    ss = -train.height_mm.values
    if yy.sum() == 0 or yy.sum() == len(yy):
        thr = THRESH_MM
    else:
        thr_score = U.best_f1_threshold(yy, ss, default=-THRESH_MM)
        thr = -float(thr_score)
    thresholds.append(thr)
    flagged = val.height_mm.values <= thr
    loo_rows.append(
        {
            "study_id": study,
            "threshold_mm": thr,
            "n": len(val),
            "pos": int(val.femur_roi.iloc[0]),
            "pred": int(flagged.any()),
        }
    )
print(pd.Series(thresholds).describe())
print("LOO studies:", len(loo_rows))


count     77.000000
mean     219.368182
std        0.504317
min      216.300000
25%      219.450000
50%      219.450000
75%      219.450000
max      219.450000
dtype: float64
LOO studies: 77


## Сохранение


In [5]:
df[
    ["file_path", "study_id", "height", "height_mm", "femur_side", "femur_roi", "comment"]
].to_csv("roi_height_validation_samples.csv", index=False)
print("saved -> roi_height_validation_samples.csv")


saved -> roi_height_validation_samples.csv


## Ошибки leave-one-study-out

Порог для каждого исследования учится без него, ошибки считаются на его снимках. Это диагностика, не независимая клиническая оценка.


In [6]:
loo_y = []
loo_pred = []
for study, val in df.groupby("study_id"):
    train = df[df.study_id != study]
    yy = train.femur_roi.values
    ss = -train.height_mm.values
    if yy.sum() > 0 and yy.sum() < len(yy):
        thr_score = U.best_f1_threshold(yy, ss, default=-THRESH_MM)
    else:
        thr_score = -THRESH_MM
    thr_mm = -float(thr_score)
    pp = (val.height_mm.values <= thr_mm).astype(int)
    loo_y.extend(val.femur_roi.values.tolist())
    loo_pred.extend(pp.tolist())

loo_y = np.asarray(loo_y, dtype=int)
loo_pred = np.asarray(loo_pred, dtype=int)
print("LOO image-level metrics:", U.binary_metrics(loo_y, loo_pred))
print(
    "LOO precision:",
    precision_score(loo_y, loo_pred, zero_division=0),
    "recall:",
    recall_score(loo_y, loo_pred, zero_division=0),
    "F1:",
    f1_score(loo_y, loo_pred, zero_division=0),
)


LOO image-level metrics: {'sens': 0.42857142857142855, 'spec': 0.9858156028368794, 'precision': 0.6, 'f1': 0.5, 'bal_acc': np.float64(0.707193515704154), 'tp': 3, 'fp': 2, 'fn': 4, 'tn': 139}
LOO precision: 0.6 recall: 0.42857142857142855 F1: 0.5
